In [ ]:
# notebooks/02_pipeline_test.ipynb

import pandas as pd
from torch.utils.data import DataLoader

from gridpulse.data.download_ett import download_ett
from gridpulse.data.validate_schema import validate_ett
from gridpulse.preprocessing.cleaning import clean_ett
from gridpulse.features.feature_builder import build_features
from gridpulse.data.split_time_series import split_by_time
from gridpulse.preprocessing.scaling import ScalerWrapper
from gridpulse.datasets.forecasting_dataset import ForecastingDataset
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1. Load + validate
df = pd.read_csv("../data/raw/ett/ETTh1.csv")
assert validate_ett(df)

In [ ]:
# 2. Clean
df = clean_ett(df)

In [ ]:
# 3. Feature engineering
df = build_features(df, target_col="OT")

In [ ]:
# 4. Split (chronological)
train_df, val_df, test_df = split_by_time(df)

In [ ]:
# 5. Scale (fit on train only!)
feature_cols = [c for c in train_df.columns if c != "date"]
scaler = ScalerWrapper(StandardScaler(), feature_cols)
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

In [ ]:
# 6. Create datasets
train_data = train_scaled[feature_cols].values
train_dataset = ForecastingDataset(train_data, input_len=96, forecast_horizon=24)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
# 7. Verify shapes
X_batch, y_batch = next(iter(train_loader))
print(f"X shape: {X_batch.shape}")  # (32, 96, num_features)
print(f"y shape: {y_batch.shape}")  # (32, 24)